In [ ]:
from lume_cheetah import LUMECheetahModel,CheetahSimulator
from virtual_accelerator.cheetah.transformer import SLACCheetahTransformer
from virtual_accelerator.cheetah.generator import generate_variables_and_mapping, split_control_and_observable
from cheetah.accelerator import Segment
from cheetah.particles import ParticleBeam
import torch
import os

# Get path to beam distributions
beam_dist = os.environ.get(
    'BEAM_DISTRIBUTION',
    '/sdf/group/ad/sw/machine-learning/Linac-Simulation-Server/simulation_server/beams'
)
# Create Cheetah particle Beam from file
incoming_beam = ParticleBeam.from_openpmd_file(
    path=os.path.join(beam_dist, "impact_inj_output_YAG03.h5"),
    energy=torch.tensor(64e6),
    dtype=torch.float32,
)
incoming_beam.particle_charges = torch.tensor(1.0)

# Get path to lattice files
lcls_lattice = os.environ.get(
    'LCLS_LATTICE',
    '/sdf/group/ad/sw/scm/repos/optics/lcls-lattice/cheetah'
)
# Create lattice from file
segment = Segment.from_lattice_json(
    os.path.join(lcls_lattice, "nc_hxr.json")
)

#Set end destination from full lattice
segment = segment.subcell(end='otr2')

# Define the simulator using lattice and particle beam
simulator = CheetahSimulator(
    segment=segment,
    initial_beam_distribution=incoming_beam,
)

# Get supported control system variables
# and a control system device to cheetah mapping
# for the model
variables, control_name_to_cheetah = generate_variables_and_mapping(segment)
#
# Create transformer that handles maps get/set calls
transformer = SLACCheetahTransformer(control_name_to_cheetah=control_name_to_cheetah)

# Define the controllable and observable variables
control_variables, observable_variables = split_control_and_observable(variables)

# Create model
model = LUMECheetahModel(
    simulator=simulator,
    transformer=transformer,
    control_variables=control_variables,
    observable_variables=observable_variables,
)

/sdf/home/c/cgarnier/.conda/envs/linac-simulation/lib/python3.11/site-packages/torch/nn/modules/module.py:2066: PhysicsWarning: Invalid tracking method 'cheetah' for element dl00 of type Drift, supported methods are ['linear', 'second_order', 'drift_kick_drift']. Keeping the previous tracking method linear.
  super().__setattr__(name, value)
/sdf/home/c/cgarnier/.conda/envs/linac-simulation/lib/python3.11/site-packages/torch/nn/modules/module.py:2066: PhysicsWarning: Invalid tracking method 'cheetah' for element loadlock of type Drift, supported methods are ['linear', 'second_order', 'drift_kick_drift']. Keeping the previous tracking method linear.
  super().__setattr__(name, value)
/sdf/home/c/cgarnier/.conda/envs/linac-simulation/lib/python3.11/site-packages/torch/nn/modules/module.py:2066: PhysicsWarning: Invalid tracking method 'cheetah' for element dl01a of type Drift, supported methods are ['linear', 'second_order', 'drift_kick_drift']. Keeping the previous tracking method linear

In [2]:
control_name_to_cheetah

{'XCOR:IN20:221': 'xc01',
 'YCOR:IN20:222': 'yc01',
 'BPMS:IN20:221': 'bpm2',
 'BPMS:IN20:235': 'bpm3',
 'QUAD:IN20:361': 'qa01',
 'QUAD:IN20:371': 'qa02',
 'QUAD:IN20:425': 'qe01',
 'BPMS:IN20:425': 'bpm6',
 'QUAD:IN20:441': 'qe02',
 'OTRS:IN20:465': 'otrh1',
 'OTRS:IN20:471': 'otrh2',
 'QUAD:IN20:511': 'qe03',
 'BPMS:IN20:511': 'bpm8',
 'XCOR:IN20:521': 'xc07',
 'YCOR:IN20:522': 'yc07',
 'QUAD:IN20:525': 'qe04',
 'BPMS:IN20:525': 'bpm9',
 'OTRS:IN20:541': 'otr1',
 'OTRS:IN20:571': 'otr2'}

## Testing getting and setting control model variables 

In [3]:
# Get variables objects
vars = list(model.supported_variables.keys())
model.supported_variables

NameError: name 'model' is not defined

In [ ]:
# Get variable values
model.get(vars)

{'XCOR:IN20:221:BCTRL': tensor(0.),
 'YCOR:IN20:222:BCTRL': tensor(0.),
 'QUAD:IN20:361:BCTRL': tensor(-4.3418),
 'QUAD:IN20:371:BCTRL': tensor(4.7358),
 'QUAD:IN20:425:BCTRL': tensor(-0.1907),
 'QUAD:IN20:441:BCTRL': tensor(0.5444),
 'QUAD:IN20:511:BCTRL': tensor(8.0429),
 'XCOR:IN20:521:BCTRL': tensor(0.),
 'YCOR:IN20:522:BCTRL': tensor(0.),
 'QUAD:IN20:525:BCTRL': tensor(-4.7208),
 'XCOR:IN20:221:BACT': tensor(0.),
 'XCOR:IN20:221:BMIN': -100.0,
 'XCOR:IN20:221:BMAX': 100.0,
 'YCOR:IN20:222:BACT': tensor(0.),
 'YCOR:IN20:222:BMIN': -100.0,
 'YCOR:IN20:222:BMAX': 100.0,
 'BPMS:IN20:221:X': tensor(-5.5928e-09),
 'BPMS:IN20:221:Y': tensor(-1.3590e-09),
 'BPMS:IN20:235:X': tensor(-5.9597e-09),
 'BPMS:IN20:235:Y': tensor(-1.4289e-09),
 'QUAD:IN20:361:BACT': tensor(-4.3418),
 'QUAD:IN20:361:BMIN': -100.0,
 'QUAD:IN20:361:BMAX': 100.0,
 'QUAD:IN20:371:BACT': tensor(4.7358),
 'QUAD:IN20:371:BMIN': -100.0,
 'QUAD:IN20:371:BMAX': 100.0,
 'BPMS:IN20:425:X': tensor(2.1319e-10),
 'BPMS:IN20:425:

In [ ]:
# Set controllable variables and verify their values changed
ctrl_vars = list(control_variables.keys())
new_values = dict(zip(ctrl_vars,[1]*len(ctrl_vars)))
model.set(new_values)
model.get(vars)

{'XCOR:IN20:221:BCTRL': tensor(1.),
 'YCOR:IN20:222:BCTRL': tensor(1.),
 'QUAD:IN20:361:BCTRL': tensor(1.0000),
 'QUAD:IN20:371:BCTRL': tensor(1.0000),
 'QUAD:IN20:425:BCTRL': tensor(1.),
 'QUAD:IN20:441:BCTRL': tensor(1.),
 'QUAD:IN20:511:BCTRL': tensor(1.),
 'XCOR:IN20:521:BCTRL': tensor(1.),
 'YCOR:IN20:522:BCTRL': tensor(1.),
 'QUAD:IN20:525:BCTRL': tensor(1.),
 'XCOR:IN20:221:BACT': tensor(1.),
 'XCOR:IN20:221:BMIN': -100.0,
 'XCOR:IN20:221:BMAX': 100.0,
 'YCOR:IN20:222:BACT': tensor(1.),
 'YCOR:IN20:222:BMIN': -100.0,
 'YCOR:IN20:222:BMAX': 100.0,
 'BPMS:IN20:221:X': tensor(0.0273),
 'BPMS:IN20:221:Y': tensor(0.0273),
 'BPMS:IN20:235:X': tensor(0.1791),
 'BPMS:IN20:235:Y': tensor(0.1791),
 'QUAD:IN20:361:BACT': tensor(1.0000),
 'QUAD:IN20:361:BMIN': -100.0,
 'QUAD:IN20:361:BMAX': 100.0,
 'QUAD:IN20:371:BACT': tensor(1.0000),
 'QUAD:IN20:371:BMIN': -100.0,
 'QUAD:IN20:371:BMAX': 100.0,
 'BPMS:IN20:425:X': tensor(0.1103),
 'BPMS:IN20:425:Y': tensor(3.8180),
 'QUAD:IN20:425:BACT': t

In [ ]:
# Reset model and verify variables when back to init values
model.reset()
model.get(vars)

{'XCOR:IN20:221:BCTRL': tensor(0.),
 'YCOR:IN20:222:BCTRL': tensor(0.),
 'QUAD:IN20:361:BCTRL': tensor(-4.3418),
 'QUAD:IN20:371:BCTRL': tensor(4.7358),
 'QUAD:IN20:425:BCTRL': tensor(-0.1907),
 'QUAD:IN20:441:BCTRL': tensor(0.5444),
 'QUAD:IN20:511:BCTRL': tensor(8.0429),
 'XCOR:IN20:521:BCTRL': tensor(0.),
 'YCOR:IN20:522:BCTRL': tensor(0.),
 'QUAD:IN20:525:BCTRL': tensor(-4.7208),
 'XCOR:IN20:221:BACT': tensor(0.),
 'XCOR:IN20:221:BMIN': -100.0,
 'XCOR:IN20:221:BMAX': 100.0,
 'YCOR:IN20:222:BACT': tensor(0.),
 'YCOR:IN20:222:BMIN': -100.0,
 'YCOR:IN20:222:BMAX': 100.0,
 'BPMS:IN20:221:X': tensor(-5.5928e-09),
 'BPMS:IN20:221:Y': tensor(-1.3590e-09),
 'BPMS:IN20:235:X': tensor(-5.9597e-09),
 'BPMS:IN20:235:Y': tensor(-1.4289e-09),
 'QUAD:IN20:361:BACT': tensor(-4.3418),
 'QUAD:IN20:361:BMIN': -100.0,
 'QUAD:IN20:361:BMAX': 100.0,
 'QUAD:IN20:371:BACT': tensor(4.7358),
 'QUAD:IN20:371:BMIN': -100.0,
 'QUAD:IN20:371:BMAX': 100.0,
 'BPMS:IN20:425:X': tensor(2.1319e-10),
 'BPMS:IN20:425: